# 🔬 Stomata Detection — Faster R-CNN v3 (False Positive Fix)

**Critical fixes from v2:**
- Fixes duplicate stomata class (ID 0 + ID 2 both = stomata)
- Conservative anchor sizes (3 ratios, not 5)
- Higher RPN NMS threshold to reduce proposal spam
- Proper 2-class training (stomata + artifact only)

**v2 Problem:** Massive false positives — green boxes covering entire image

---
**⚠️ Runtime → Change runtime type → GPU (T4 or better)**

## 1. Setup

In [ ]:
import torch
print("=" * 60)
print("🖥️  GPU STATUS")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    device = torch.device('cuda')
    recommended_batch = 4 if gpu_mem >= 10 else 2
else:
    print("❌ No GPU!")
    device = torch.device('cpu')
    recommended_batch = 1

print(f"PyTorch: {torch.__version__}")
print(f"Batch size: {recommended_batch}")
print("=" * 60)

In [ ]:
!pip install roboflow pycocotools --quiet
import torchvision
print(f"✅ Torchvision: {torchvision.__version__}")

## 2. Download & Fix Dataset

**⚠️ CRITICAL:** This section fixes the duplicate stomata class issue

In [ ]:
from roboflow import Roboflow

# ┌──────────────────────────────────────────────────────────────┐
# │  ⚠️ REPLACE WITH YOUR ACTUAL CREDENTIALS                    │
# └──────────────────────────────────────────────────────────────┘
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download("coco", location="/content/stomata_dataset")

print("✅ Downloaded")

In [ ]:
# ══════════════════════════════════════════════════════════════
# AUTO-DETECT DATASET PATH
# ══════════════════════════════════════════════════════════════
import os
import json
from glob import glob

dataset_root = None
for d in ["/content/stomata_dataset"] + \
         [os.path.join("/content", x) for x in os.listdir("/content")
          if os.path.isdir(os.path.join("/content", x)) and x not in ['drive', 'sample_data', '.config']]:
    if os.path.exists(os.path.join(d, "train", "_annotations.coco.json")):
        dataset_root = d
        break

if dataset_root is None:
    found = glob("/content/**/train/_annotations.coco.json", recursive=True)
    if found:
        dataset_root = os.path.dirname(os.path.dirname(found[0]))

assert dataset_root is not None, "❌ Dataset not found!"
print(f"✅ Dataset: {dataset_root}")

train_dir = os.path.join(dataset_root, "train")
valid_dir = os.path.join(dataset_root, "valid")
test_dir = os.path.join(dataset_root, "test")
train_anno = os.path.join(train_dir, "_annotations.coco.json")
valid_anno = os.path.join(valid_dir, "_annotations.coco.json")
test_anno = os.path.join(test_dir, "_annotations.coco.json")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🔴 CRITICAL FIX: MERGE DUPLICATE CLASSES
# Your dataset has: ID 0=stomata, ID 1=artifact, ID 2=stomata
# This causes massive false positives!
# ══════════════════════════════════════════════════════════════

print("=" * 60)
print("🔧 FIXING DUPLICATE CLASSES")
print("=" * 60)

for split in ['train', 'valid', 'test']:
    anno_path = os.path.join(dataset_root, split, '_annotations.coco.json')
    if not os.path.exists(anno_path):
        print(f"  ⏭️ [{split}] Not found, skipping")
        continue

    with open(anno_path, 'r') as f:
        coco = json.load(f)

    # Show BEFORE
    print(f"\n  [{split}] BEFORE: {len(coco['categories'])} categories")
    for cat in coco['categories']:
        count = sum(1 for a in coco['annotations'] if a['category_id'] == cat['id'])
        print(f"    ID {cat['id']}: '{cat['name']}' ({count} annotations)")

    # Find duplicates by normalized name
    seen_names = {}  # normalized_name -> first category_id
    id_remap = {}    # old_id -> new_id
    clean_categories = []

    for cat in coco['categories']:
        name_key = cat['name'].lower().strip()
        if name_key in seen_names:
            # DUPLICATE! Remap to first occurrence
            id_remap[cat['id']] = seen_names[name_key]
            print(f"    🔀 Merging '{cat['name']}' ID {cat['id']} → ID {seen_names[name_key]}")
        else:
            seen_names[name_key] = cat['id']
            clean_categories.append(cat)

    # Apply remapping
    if id_remap:
        remapped_count = 0
        for ann in coco['annotations']:
            if ann['category_id'] in id_remap:
                ann['category_id'] = id_remap[ann['category_id']]
                remapped_count += 1

        coco['categories'] = clean_categories

        # Save fixed file
        with open(anno_path, 'w') as f:
            json.dump(coco, f)

        print(f"    ✅ FIXED! Remapped {remapped_count} annotations")
    else:
        print(f"    ✅ No duplicates found")

    # Show AFTER
    print(f"  [{split}] AFTER: {len(coco['categories'])} categories")
    for cat in coco['categories']:
        count = sum(1 for a in coco['annotations'] if a['category_id'] == cat['id'])
        print(f"    ID {cat['id']}: '{cat['name']}' ({count} annotations)")

print("\n" + "=" * 60)
print("✅ ANNOTATION FIX COMPLETE")
print("=" * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════
# VERIFY FIXED DATASET
# ══════════════════════════════════════════════════════════════

with open(train_anno, 'r') as f:
    train_coco = json.load(f)

categories = train_coco['categories']
print(f"\n📋 Final classes ({len(categories)}):")
for cat in categories:
    count = sum(1 for a in train_coco['annotations'] if a['category_id'] == cat['id'])
    print(f"   ID {cat['id']}: {cat['name']} — {count} annotations")

assert len(categories) == 2, f"❌ Expected 2 classes, got {len(categories)}. Fix failed!"
print(f"\n✅ Correct: 2 classes (+ background = 3 total for Faster R-CNN)")

# Identify stomata/artifact IDs
stomata_cat_id = None
artifact_cat_id = None
cat_id_to_name = {}

for cat in categories:
    cat_id_to_name[cat['id']] = cat['name']
    name_lower = cat['name'].lower()
    if 'stomata' in name_lower or 'stoma' in name_lower:
        stomata_cat_id = cat['id']
    elif 'artifact' in name_lower or 'bubble' in name_lower:
        artifact_cat_id = cat['id']

print(f"🎯 Stomata: ID {stomata_cat_id}")
print(f"🚫 Artifact: ID {artifact_cat_id}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BBOX SIZE ANALYSIS
# ══════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import random

widths = [a['bbox'][2] for a in train_coco['annotations']]
heights = [a['bbox'][3] for a in train_coco['annotations']]

print("📏 Bounding Box Sizes:")
print(f"   Width  — Median: {np.median(widths):.0f}  Mean: {np.mean(widths):.0f}  Range: {min(widths):.0f}-{max(widths):.0f}")
print(f"   Height — Median: {np.median(heights):.0f}  Mean: {np.mean(heights):.0f}  Range: {min(heights):.0f}-{max(heights):.0f}")

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(widths, heights, alpha=0.3, s=10, color='green')
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.set_title('Stomata Box Sizes — anchor sizes should cover this range')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import cv2


class StomataDataset(Dataset):
    def __init__(self, root_dir, annotation_file, train=False):
        self.root_dir = root_dir
        self.train = train

        with open(annotation_file, 'r') as f:
            coco = json.load(f)

        self.images = coco['images']
        self.categories = coco['categories']

        self.cat_id_to_label = {}
        for i, cat in enumerate(self.categories):
            self.cat_id_to_label[cat['id']] = i + 1  # 0=background

        self.label_to_name = {0: 'background'}
        for cat in self.categories:
            self.label_to_name[self.cat_id_to_label[cat['id']]] = cat['name']

        self.img_annotations = {}
        for ann in coco['annotations']:
            iid = ann['image_id']
            if iid not in self.img_annotations:
                self.img_annotations[iid] = []
            self.img_annotations[iid].append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.root_dir, img_info['file_name'])
        img = Image.open(img_path).convert('RGB')

        annos = self.img_annotations.get(img_info['id'], [])
        boxes, labels, areas, iscrowd = [], [], [], []

        for ann in annos:
            x, y, w, h = ann['bbox']
            if w <= 2 or h <= 2:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(self.cat_id_to_label[ann['category_id']])
            areas.append(ann.get('area', w * h))
            iscrowd.append(ann.get('iscrowd', 0))

        # Augmentation: horizontal flip
        if self.train and len(boxes) > 0 and random.random() > 0.5:
            img = TF.hflip(img)
            w_img = img.width
            boxes = [[w_img - x2, y1, w_img - x1, y2] for x1, y1, x2, y2 in boxes]

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            areas = torch.as_tensor(areas, dtype=torch.float32)
            iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64)

        target = {
            'boxes': boxes, 'labels': labels,
            'image_id': torch.tensor([img_info['id']]),
            'area': areas, 'iscrowd': iscrowd
        }

        img = TF.to_tensor(img)

        # Light color augmentation
        if self.train:
            if random.random() > 0.6:
                img = TF.adjust_brightness(img, random.uniform(0.9, 1.1))
            if random.random() > 0.6:
                img = TF.adjust_contrast(img, random.uniform(0.9, 1.1))

        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = StomataDataset(train_dir, train_anno, train=True)
valid_dataset = StomataDataset(valid_dir, valid_anno, train=False)

train_loader = DataLoader(train_dataset, batch_size=recommended_batch,
                          shuffle=True, num_workers=2,
                          collate_fn=collate_fn, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=recommended_batch,
                          shuffle=False, num_workers=2,
                          collate_fn=collate_fn, pin_memory=True)

num_classes = len(train_dataset.categories) + 1
print(f"\n✅ Classes: {num_classes} → {train_dataset.label_to_name}")
print(f"   Train: {len(train_dataset)} | Valid: {len(valid_dataset)}")

# Sanity check
assert num_classes == 3, f"❌ Expected 3 classes (bg+stomata+artifact), got {num_classes}"

## 4. Build Model (Conservative — Anti False Positive)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONSERVATIVE MODEL — FIXES FALSE POSITIVE FLOOD
# ══════════════════════════════════════════════════════════════
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


def build_model(num_classes):
    """
    Conservative Faster R-CNN — tuned to REDUCE false positives.
    Uses DEFAULT anchors (3 ratios) — the v2 custom anchors caused FP flood.
    """
    model = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=800,
        max_size=800,
        # ANTI-FP: Higher score threshold filters weak detections
        box_score_thresh=0.05,
        # ANTI-FP: Stricter NMS removes more overlapping boxes
        box_nms_thresh=0.4,
        box_detections_per_img=200,
        # RPN: fewer proposals = fewer false positives
        rpn_pre_nms_top_n_train=2000,
        rpn_post_nms_top_n_train=1000,
        rpn_pre_nms_top_n_test=1000,
        rpn_post_nms_top_n_test=500,
        rpn_nms_thresh=0.7,
    )

    # Replace head for our classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Freeze backbone layers 1-2, fine-tune 3-4
    for name, param in model.backbone.body.named_parameters():
        if 'layer1' in name or 'layer2' in name:
            param.requires_grad = False

    return model


model = build_model(num_classes)
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Model: Faster R-CNN ResNet50-FPN v2")
print(f"   Trainable: {trainable:,} / {total:,}")
print(f"   Using DEFAULT anchors (3 aspect ratios)")
print(f"   NMS thresh: 0.4 (stricter than default 0.5)")

## 5. Train

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/Stomata_FRCNN_v3'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


@torch.no_grad()
def evaluate_model(model, data_loader, device, anno_file, dataset_obj):
    model.eval()
    coco_gt = COCO(anno_file)
    label_to_cat_id = {v: k for k, v in dataset_obj.cat_id_to_label.items()}
    results = []

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        for target, output in zip(targets, outputs):
            image_id = target['image_id'].item()
            for box, score, label in zip(
                output['boxes'].cpu().numpy(),
                output['scores'].cpu().numpy(),
                output['labels'].cpu().numpy()
            ):
                if label == 0:
                    continue
                x1, y1, x2, y2 = box
                results.append({
                    'image_id': image_id,
                    'category_id': int(label_to_cat_id.get(label, label)),
                    'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                    'score': float(score)
                })

    if not results:
        return 0.0, 0.0

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval.stats[1], coco_eval.stats[0]  # mAP50, mAP50-95


print("✅ Evaluator ready")

In [ ]:
# ══════════════════════════════════════════════════════════════
#                 🚀 TRAINING LOOP
# ══════════════════════════════════════════════════════════════
import time

NUM_EPOCHS = 100
PATIENCE = 25
EVAL_EVERY = 5

# Differential LR: lower for backbone, higher for head
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' not in n]

optimizer = torch.optim.SGD([
    {'params': backbone_params, 'lr': 0.0002},
    {'params': head_params, 'lr': 0.005},
], momentum=0.9, weight_decay=0.0005, nesterov=True)

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

WARMUP_ITERS = len(train_loader) * 3

history = {'train_loss': [], 'mAP_50': [], 'mAP_50_95': [], 'lr': []}
best_mAP = 0.0
best_epoch = 0
patience_counter = 0
global_step = 0

print("=" * 60)
print("🚀 TRAINING v3 (duplicate class fixed, conservative anchors)")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    t0 = time.time()

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Warmup
        if global_step < WARMUP_ITERS:
            alpha = float(global_step) / WARMUP_ITERS
            for pg in optimizer.param_groups:
                pg['lr'] = pg['lr'] * (0.001 * (1 - alpha) + alpha)

        with torch.amp.autocast('cuda'):
            loss_dict = model(images, targets)
            total_loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=10.0
        )
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += total_loss.item()
        num_batches += 1
        global_step += 1

    avg_loss = epoch_loss / num_batches
    current_lr = optimizer.param_groups[1]['lr']
    history['train_loss'].append(avg_loss)
    history['lr'].append(current_lr)

    print(f"Epoch [{epoch:3d}/{NUM_EPOCHS}]  Loss: {avg_loss:.4f}  LR: {current_lr:.6f}  {time.time()-t0:.0f}s")

    if epoch % EVAL_EVERY == 0 or epoch == 1:
        print(f"  📊 Evaluating...")
        mAP_50, mAP_50_95 = evaluate_model(model, valid_loader, device, valid_anno, valid_dataset)
        history['mAP_50'].append(mAP_50)
        history['mAP_50_95'].append(mAP_50_95)

        print(f"  mAP@50: {mAP_50:.4f} ({mAP_50*100:.1f}%) | mAP@50-95: {mAP_50_95:.4f}")

        if mAP_50 > best_mAP:
            best_mAP = mAP_50
            best_mAP_95 = mAP_50_95
            best_epoch = epoch
            patience_counter = 0

            best_save = {
                'model_state_dict': model.state_dict(),
                'epoch': epoch,
                'mAP_50': mAP_50,
                'mAP_50_95': mAP_50_95,
                'num_classes': num_classes,
                'label_to_name': train_dataset.label_to_name,
                'cat_id_to_label': train_dataset.cat_id_to_label,
                'categories': train_dataset.categories,
                'img_size': 800,
                'model_type': 'faster_rcnn_resnet50_fpn_v2',
                # No custom anchor_sizes key = app uses default anchors
            }
            torch.save(best_save, '/content/best_stomata_frcnn.pt')
            torch.save(best_save, os.path.join(save_dir, 'best.pt'))
            print(f"  🏆 NEW BEST! Saved.")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{PATIENCE//EVAL_EVERY})")

        if patience_counter >= (PATIENCE // EVAL_EVERY):
            print(f"\n⏹️ Early stop. Best: epoch {best_epoch} (mAP@50: {best_mAP:.4f})")
            break
        print()
    else:
        history['mAP_50'].append(None)
        history['mAP_50_95'].append(None)

    if epoch > 3:
        lr_scheduler.step()

print(f"\n✅ DONE! Best: mAP@50={best_mAP:.4f} at epoch {best_epoch}")

## 6. Results & Predictions

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], 'b-', linewidth=2)
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].grid(True, alpha=0.3)

eval_ep = [i+1 for i, v in enumerate(history['mAP_50']) if v is not None]
eval_50 = [v for v in history['mAP_50'] if v is not None]
eval_95 = [v for v in history['mAP_50_95'] if v is not None]

axes[1].plot(eval_ep, eval_50, 'g-o', linewidth=2, label='mAP@50')
axes[1].plot(eval_ep, eval_95, 'r-s', linewidth=2, label='mAP@50-95')
axes[1].set_title('Validation mAP', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

axes[2].plot(history['lr'], 'purple', linewidth=2)
axes[2].set_title('Learning Rate', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'v3 Training — Best mAP@50: {best_mAP:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'training_curves_v3.png'), dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# LOAD BEST & PREDICT
# ══════════════════════════════════════════════════════════════

best_ckpt = torch.load('/content/best_stomata_frcnn.pt',
                       map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()

label_to_name = best_ckpt['label_to_name']
stomata_label = None
for lbl, name in label_to_name.items():
    if lbl != 0 and 'stomata' in name.lower():
        stomata_label = lbl
        break

print(f"✅ Loaded best (epoch {best_ckpt['epoch']}, mAP@50: {best_ckpt['mAP_50']:.4f})")
print(f"   Stomata label: {stomata_label}")

In [ ]:
@torch.no_grad()
def predict_stomata(model, image_path, conf=0.3, iou_thresh=0.4, device='cuda'):
    """Predict with higher confidence threshold to avoid false positives."""
    model.eval()
    img_cv = cv2.imread(image_path)
    h, w = img_cv.shape[:2]
    img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    img_tensor = TF.to_tensor(Image.fromarray(img_rgb)).to(device)

    output = model([img_tensor])[0]
    keep = torchvision.ops.nms(output['boxes'], output['scores'],
                               iou_threshold=iou_thresh).cpu().numpy()

    annotated = img_cv.copy()
    stomata_count = 0

    for idx in keep:
        score = output['scores'][idx].item()
        if score < conf:
            continue

        label = int(output['labels'][idx].item())
        if label != stomata_label:
            continue

        x1, y1, x2, y2 = map(int, output['boxes'][idx].cpu().numpy())

        # Skip black padding
        cx, cy = (x1+x2)//2, (y1+y2)//2
        if 0 <= cy < h and 0 <= cx < w and np.mean(img_cv[cy, cx]) < 15:
            continue
        if (x2-x1) < 10 or (y2-y1) < 10:
            continue

        stomata_count += 1
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(annotated, f"{score:.2f}", (x1, y1-5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    return stomata_count, annotated


# Visualize on validation set
val_images = sorted(glob(os.path.join(valid_dir, "*.jpg")) +
                    glob(os.path.join(valid_dir, "*.jpeg")) +
                    glob(os.path.join(valid_dir, "*.png")))

samples = random.sample(val_images, min(8, len(val_images)))
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

for ax, img_path in zip(axes.flatten(), samples):
    count, ann = predict_stomata(model, img_path, conf=0.3, device=device)
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Stomata: {count}", fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('v3 Predictions (conf=0.3) — Should be MUCH cleaner than v2',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'predictions_v3.png'), dpi=150)
plt.show()

## 7. Test Set

In [ ]:
if os.path.exists(test_anno):
    test_dataset = StomataDataset(test_dir, test_anno, train=False)
    test_loader = DataLoader(test_dataset, batch_size=recommended_batch,
                             shuffle=False, num_workers=2, collate_fn=collate_fn)

    print("Evaluating on TEST set...")
    test_50, test_95 = evaluate_model(model, test_loader, device, test_anno, test_dataset)

    print("\n" + "=" * 60)
    print("🧪 TEST RESULTS v3")
    print("=" * 60)
    print(f"mAP@50:    {test_50:.4f} ({test_50*100:.1f}%)")
    print(f"mAP@50-95: {test_95:.4f} ({test_95*100:.1f}%)")
    print("=" * 60)

## 8. Export for Desktop App

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

final_model = {
    'model_state_dict': best_ckpt['model_state_dict'],
    'num_classes': best_ckpt['num_classes'],
    'label_to_name': best_ckpt['label_to_name'],
    'cat_id_to_label': best_ckpt['cat_id_to_label'],
    'categories': best_ckpt['categories'],
    'img_size': 800,
    'model_type': 'faster_rcnn_resnet50_fpn_v2',
    'mAP_50': best_ckpt['mAP_50'],
    'mAP_50_95': best_ckpt['mAP_50_95'],
    'training_date': timestamp,
    # NO anchor_sizes key → desktop app uses default anchors ✓
}

torch.save(final_model, os.path.join(save_dir, 'best_stomata_frcnn.pt'))
torch.save(final_model, os.path.join(save_dir, f'best_frcnn_v3_{timestamp}.pt'))

print(f"\n✅ Saved to: {save_dir}")
print(f"\n🖥️ Load best_stomata_frcnn.pt in stomata_analyzer_v8_dual_model.py")
print(f"   The app auto-detects Faster R-CNN and uses default anchors")

---
## Summary: What Fixed the False Positives

| Problem | Root Cause | Fix |
|---------|-----------|-----|
| Green boxes flooding entire image | Duplicate stomata class (ID 0 + ID 2) | Merge duplicates before training |
| Too many overlapping detections | 5 aspect ratios = too many anchors | Back to default 3 ratios |
| Low confidence detections counted | conf threshold 0.25 too permissive | Use 0.30+ for inference |
| Large boxes covering regions | Custom anchor sizes too aggressive | Use default COCO anchors |

**Tip:** In the desktop app, start with confidence = **0.30** and IoU = **0.40** for this model.